# Benchmark running and visualization
1. Build the benchmarks, using `BOOST_PROMETHEUS_BENCH` CMake option and the `boost_prometheus_bench` CMake target.
2. Run notebooks to benchmark and plot graphs.

In [ ]:
import os
import plotly.io as pio

pio.renderers.default = "plotly_mimetype+png"

# Path to the binaries.
bench_path = os.path.abspath(os.path.join(os.getcwd(), "..", "build", "bench"))

#### **histogram_observe**

In [ ]:
from plotly.subplots import make_subplots
from bench_utils import run_benchmark, load_pair, thread_stats, add_metadata_footer, make_table, add_line_traces

df, meta = load_pair(run_benchmark, bench_path,
    "boost_prometheus_bench_histogram_observe_boost",
    "boost_prometheus_bench_histogram_observe_prometheus-cpp")

fig = make_subplots(
    rows=2, cols=1,
    row_heights=[0.62, 0.38],
    vertical_spacing=0.14,
    specs=[[{"type": "xy"}], [{"type": "table"}]],
)

add_line_traces(fig, df, row=1, col=1)
fig.add_trace(make_table(*thread_stats(df)), row=2, col=1)

fig.update_xaxes(title_text="Threads", row=1, col=1)
fig.update_yaxes(title_text="M ops/s", row=1, col=1)

add_metadata_footer(fig, meta)

fig.update_layout(
    title="Histogram Observe Performance",
    height=820, width=1000, hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=1.06,
                xanchor="center", x=0.5),
    margin=dict(l=60, r=60, t=90, b=90),
)
fig.show()

#### **metric_vec (family) search**

In [ ]:
from plotly.subplots import make_subplots
from bench_utils import run_benchmark, load_pair, thread_stats, add_metadata_footer, make_table, add_line_traces

df, meta = load_pair(run_benchmark, bench_path,
    "boost_prometheus_bench_metric_vec_boost",
    "boost_prometheus_bench_metric_vec_prometheus-cpp")

fig = make_subplots(
    rows=2, cols=2,
    row_heights=[0.62, 0.38],
    vertical_spacing=0.14,
    subplot_titles=("metric_vec Lookup Performance (2 Labels)",
                    "metric_vec Lookup Performance (4 Labels)"),
    specs=[[{"type": "xy"}, {"type": "xy"}],
           [{"type": "table"}, {"type": "table"}]],
)

for col_idx, num_labels in enumerate([2, 4], start=1):
    data = df[df["labels"] == num_labels]
    add_line_traces(fig, data, row=1, col=col_idx, showlegend=(col_idx == 1))
    fig.update_xaxes(title_text="Threads", row=1, col=col_idx)
    fig.update_yaxes(title_text="M ops/s", row=1, col=col_idx)

    stats = thread_stats(df, num_labels=num_labels)
    fig.add_trace(make_table(*stats), row=2, col=col_idx)

add_metadata_footer(fig, meta)

fig.update_layout(
    height=900, width=1200, hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=1.06,
                xanchor="center", x=0.5),
    margin=dict(l=60, r=60, t=90, b=90),
)
fig.show()

#### **registry serialization**

In [ ]:
from plotly.subplots import make_subplots
from bench_utils import run_benchmark, load_pair, add_metadata_footer, make_table, add_line_traces_metric, metric_stats

df, meta = load_pair(run_benchmark, bench_path,
    "boost_prometheus_bench_serialization_boost",
    "boost_prometheus_bench_serialization_prometheus-cpp")

fig = make_subplots(
    rows=2, cols=1,
    row_heights=[0.62, 0.38],
    vertical_spacing=0.14,
    specs=[[{"type": "xy"}], [{"type": "table"}]],
)

add_line_traces_metric(fig, df, row=1, col=1)

counts, prom_ops, boost_ops, diffs = metric_stats(df)
fig.add_trace(
    make_table(counts, prom_ops, boost_ops, diffs, x_header="Metrics"),
    row=2, col=1,
)

fig.update_xaxes(title_text="Metrics", type="log", row=1, col=1)
fig.update_yaxes(title_text="K ops/s", row=1, col=1)

add_metadata_footer(fig, meta)

fig.update_layout(
    title="Registration + Serialization Performance",
    height=720, width=1000, hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=1.06,
                xanchor="center", x=0.5),
    margin=dict(l=60, r=60, t=90, b=90),
)
fig.show()